# HW13

Домашняя работа по токенизации текста, инференсу готовой BERT-подобной модели и базовому fine-tuning для классификации текста.


## 1. Импорты, seed и среда


In [1]:
import json
import os
import platform
import random
from pathlib import Path

import datasets
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from datasets import load_dataset
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
)

PROJECT_DIR = Path.cwd()
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = '/tmp/hf'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['MPLCONFIGDIR'] = str(ARTIFACTS_DIR / 'mplconfig')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FINETUNE_MODEL_NAME = 'google/bert_uncased_L-2_H-128_A-2'
INFERENCE_MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 5e-5
EPOCHS = 5
EXPERIMENT_SIZES = {'train': 4000, 'validation': 1000, 'test': 1000}

versions_df = pd.DataFrame(
    {'value': [
        platform.python_version(),
        datasets.__version__,
        transformers.__version__,
        torch.__version__,
        np.__version__,
        pd.__version__,
        sklearn.__version__,
        str(DEVICE),
        SEED,
    ]},
    index=['python', 'datasets', 'transformers', 'torch', 'numpy', 'pandas', 'sklearn', 'device', 'seed'],
)
versions_df


/home/candy/Рабочий стол/aie-group/.venv-hw13/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,value
python,3.14.3
datasets,4.8.4
transformers,5.4.0
torch,2.10.0
numpy,2.4.3
pandas,2.3.3
sklearn,1.8.0
device,cpu
seed,42


## 2. Данные и первичный анализ


In [2]:
dataset = load_dataset('emotion')
label_names = dataset['train'].features['label'].names
official_sizes = {split: len(dataset[split]) for split in dataset}

split_sizes_df = pd.DataFrame({
    'split': list(official_sizes.keys()),
    'size': list(official_sizes.values()),
})
split_sizes_df


,split,size
0,train,16000
1,validation,2000
2,test,2000


In [3]:
experiment_dataset = {
    split: dataset[split].shuffle(seed=SEED).select(range(size))
    for split, size in EXPERIMENT_SIZES.items()
}

experiment_sizes_df = pd.DataFrame({
    'split': list(experiment_dataset.keys()),
    'size': [len(experiment_dataset[split]) for split in experiment_dataset],
})
experiment_sizes_df


,split,size
0,train,4000
1,validation,1000
2,test,1000


In [4]:
examples_df = pd.DataFrame(experiment_dataset['train'][:5])
examples_df['label_name'] = examples_df['label'].map(lambda idx: label_names[idx])
examples_df[['text', 'label', 'label_name']]


,text,label,label_name
0,while cycling in the country,4,fear
1,i had pocket qq and was feeling pretty confide...,1,joy
2,i am in no way complaining or whining or feeli...,0,sadness
3,i feel a bit stressed because it feels like im...,3,anger
4,i tell the people closest to me things that i ...,5,surprise


`emotion` содержит короткие пользовательские тексты, размеченные по 6 эмоциям: `sadness`, `joy`, `love`, `anger`, `fear`, `surprise`.
Для воспроизводимого CPU-запуска в эксперименте используется фиксированный поднабор официальных split-частей.


## 3. Токенизация


In [5]:
finetune_tokenizer = AutoTokenizer.from_pretrained(FINETUNE_MODEL_NAME)

tokenization_records = []
for idx, row in examples_df.iterrows():
    encoded = finetune_tokenizer(row['text'], truncation=True, max_length=32)
    tokens = finetune_tokenizer.convert_ids_to_tokens(encoded['input_ids'])
    tokenization_records.append({
        'example_id': idx + 1,
        'label_name': row['label_name'],
        'text': row['text'],
        'tokens': tokens,
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
        'special_tokens': [tokens[0], tokens[-1]],
        'sequence_length': len(encoded['input_ids']),
    })

tokenization_df = pd.DataFrame(tokenization_records)
tokenization_df[['example_id', 'label_name', 'text', 'tokens', 'input_ids', 'attention_mask', 'special_tokens']]


,example_id,label_name,text,tokens,input_ids,attention_mask,special_tokens
0,1,fear,while cycling in the country,"[[CLS], while, cycling, in, the, country, [SEP]]","[101, 2096, 9670, 1999, 1996, 2406, 102]","[1, 1, 1, 1, 1, 1, 1]","[[CLS], [SEP]]"
1,2,joy,i had pocket qq and was feeling pretty confide...,"[[CLS], i, had, pocket, q, ##q, and, was, feel...","[101, 1045, 2018, 4979, 1053, 4160, 1998, 2001...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]","[[CLS], [SEP]]"
2,3,sadness,i am in no way complaining or whining or feeli...,"[[CLS], i, am, in, no, way, complaining, or, w...","[101, 1045, 2572, 1999, 2053, 2126, 17949, 203...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[CLS], [SEP]]"
3,4,anger,i feel a bit stressed because it feels like im...,"[[CLS], i, feel, a, bit, stressed, because, it...","[101, 1045, 2514, 1037, 2978, 13233, 2138, 200...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[CLS], [SEP]]"
4,5,surprise,i tell the people closest to me things that i ...,"[[CLS], i, tell, the, people, closest, to, me,...","[101, 1045, 2425, 1996, 2111, 7541, 2000, 2033...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[CLS], [SEP]]"


In [6]:
padded_example = finetune_tokenizer(
    examples_df.loc[0, 'text'],
    truncation=True,
    padding='max_length',
    max_length=20,
)
{
    'special_tokens_map': finetune_tokenizer.special_tokens_map,
    'padded_tokens': finetune_tokenizer.convert_ids_to_tokens(padded_example['input_ids']),
    'padded_attention_mask': padded_example['attention_mask'],
}


{'special_tokens_map': {'unk_token': '[UNK]',
  'sep_token': '[SEP]',
  'pad_token': '[PAD]',
  'cls_token': '[CLS]',
  'mask_token': '[MASK]'},
 'padded_tokens': ['[CLS]',
  'while',
  'cycling',
  'in',
  'the',
  'country',
  '[SEP]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]'],
 'padded_attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]}

## 4. Инференс готовой pretrained-модели


In [7]:
inference_tokenizer = DistilBertTokenizer.from_pretrained(INFERENCE_MODEL_NAME)
inference_model = DistilBertForSequenceClassification.from_pretrained(INFERENCE_MODEL_NAME).to(DEVICE)
inference_model.eval()

inference_texts = [
    'I finally got the job and I feel amazing.',
    'I am worried that something bad is going to happen soon.',
    'She hugged me and I felt deeply cared for.',
    'I cannot believe they lied to me again.',
    'Wait, they planned a surprise party for me?',
]

inference_inputs = inference_tokenizer(
    inference_texts,
    truncation=True,
    padding=True,
    return_tensors='pt',
).to(DEVICE)

with torch.no_grad():
    inference_probs = torch.softmax(inference_model(**inference_inputs).logits, dim=-1).cpu().numpy()

inference_df = pd.DataFrame({
    'text': inference_texts,
    'predicted_label': [inference_model.config.id2label[int(row.argmax())] for row in inference_probs],
    'score': inference_probs.max(axis=1),
})
inference_df


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5374.07it/s]


,text,predicted_label,score
0,I finally got the job and I feel amazing.,POSITIVE,0.999885
1,I am worried that something bad is going to ha...,NEGATIVE,0.999586
2,She hugged me and I felt deeply cared for.,POSITIVE,0.999676
3,I cannot believe they lied to me again.,POSITIVE,0.771338
4,"Wait, they planned a surprise party for me?",NEGATIVE,0.996452


Эта модель обучена на бинарном `sentiment analysis`, поэтому она полезна как sanity-check,
но не решает напрямую задачу классификации 6 эмоций.


## 5. Fine-tuning для классификации текста


In [8]:
def tokenize_batch(batch):
    return finetune_tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

tokenized_dataset = {}
for split, split_dataset in experiment_dataset.items():
    tokenized = split_dataset.map(tokenize_batch, batched=True)
    tokenized = tokenized.rename_column('label', 'labels')
    tokenized = tokenized.remove_columns(['text'])
    tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    tokenized_dataset[split] = tokenized

data_collator = DataCollatorWithPadding(tokenizer=finetune_tokenizer)
train_loader = DataLoader(tokenized_dataset['train'], batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=data_collator)
val_loader = DataLoader(tokenized_dataset['validation'], batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test_loader = DataLoader(tokenized_dataset['test'], batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=data_collator)

model = AutoModelForSequenceClassification.from_pretrained(
    FINETUNE_MODEL_NAME,
    num_labels=len(label_names),
    ignore_mismatched_sizes=True,
).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

def move_batch_to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

def evaluate(model, loader):
    model.eval()
    losses = []
    preds = []
    gold = []
    confidences = []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch_to_device(batch)
            outputs = model(**batch)
            losses.append(outputs.loss.item())
            probabilities = torch.softmax(outputs.logits, dim=-1)
            preds.extend(torch.argmax(probabilities, dim=-1).cpu().tolist())
            gold.extend(batch['labels'].cpu().tolist())
            confidences.extend(probabilities.max(dim=-1).values.cpu().tolist())
    return {
        'loss': float(np.mean(losses)),
        'accuracy': accuracy_score(gold, preds),
        'f1_macro': f1_score(gold, preds, average='macro'),
        'preds': preds,
        'gold': gold,
        'confidences': confidences,
    }


Loading weights: 100%|██████████| 39/39 [00:00<00:00, 6608.41it/s]
BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

In [9]:
history = []
best_state = None
best_epoch = None
best_val_f1 = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for batch in progress:
        batch = move_batch_to_device(batch)
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        progress.set_postfix(train_loss=f'{loss.item():.4f}')

    val_metrics = evaluate(model, val_loader)
    history.append({
        'epoch': epoch,
        'train_loss': float(np.mean(train_losses)),
        'val_loss': val_metrics['loss'],
        'val_accuracy': val_metrics['accuracy'],
        'val_f1_macro': val_metrics['f1_macro'],
    })

    if val_metrics['f1_macro'] > best_val_f1:
        best_val_f1 = val_metrics['f1_macro']
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

history_df = pd.DataFrame(history)
history_df


Epoch 5/5: 100%|██████████| 125/125 [00:11<00:00, 10.84it/s, train_loss=0.7143]


,epoch,train_loss,val_loss,val_accuracy,val_f1_macro
0,1,1.612451,1.579654,0.367,0.089621
1,2,1.547236,1.476174,0.491,0.194263
2,3,1.344841,1.165030,0.612,0.345922
3,4,1.051226,0.911527,0.684,0.512384
4,5,0.826091,0.750106,0.783,0.703114


In [10]:
model.load_state_dict(best_state)
model.to(DEVICE)
test_metrics = evaluate(model, test_loader)

metrics_df = pd.DataFrame([{
    'dataset': 'emotion',
    'train_size': EXPERIMENT_SIZES['train'],
    'validation_size': EXPERIMENT_SIZES['validation'],
    'test_size': EXPERIMENT_SIZES['test'],
    'model_name': FINETUNE_MODEL_NAME,
    'inference_model_name': INFERENCE_MODEL_NAME,
    'max_length': MAX_LENGTH,
    'batch_size': TRAIN_BATCH_SIZE,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'best_epoch': best_epoch,
    'val_f1_macro': best_val_f1,
    'test_loss': test_metrics['loss'],
    'test_accuracy': test_metrics['accuracy'],
    'test_f1_macro': test_metrics['f1_macro'],
}])
metrics_df


,dataset,train_size,validation_size,test_size,model_name,inference_model_name,max_length,batch_size,epochs,learning_rate,best_epoch,val_f1_macro,test_loss,test_accuracy,test_f1_macro
0,emotion,4000,1000,1000,google/bert_uncased_L-2_H-128_A-2,distilbert-base-uncased-finetuned-sst-2-english,128,32,5,0.00005,5,0.703114,0.737954,0.786,0.672932


## 6. Оценка качества и анализ ошибок


In [11]:
test_rows = []
for text, true_idx, pred_idx, conf in zip(
    experiment_dataset['test']['text'],
    test_metrics['gold'],
    test_metrics['preds'],
    test_metrics['confidences'],
):
    test_rows.append({
        'text': text,
        'true_label': label_names[true_idx],
        'pred_label': label_names[pred_idx],
        'confidence': float(conf),
        'is_correct': bool(true_idx == pred_idx),
    })

predictions_df = pd.DataFrame(test_rows)
sample_predictions_df = predictions_df.sort_values(['is_correct', 'confidence'], ascending=[True, True]).head(10)
sample_predictions_df.drop(columns=['is_correct']).to_csv(ARTIFACTS_DIR / 'sample_predictions.csv', index=False)
sample_predictions_df.drop(columns=['is_correct']).reset_index(drop=True)


,text,true_label,pred_label,confidence
0,i feel irritable about the number of people th...,anger,sadness,0.251108
1,im amazed how many men say they feel unloved i...,sadness,surprise,0.272228
2,i feel a strange gratitude for the hated israe...,surprise,fear,0.281038
3,im okay with her getting married whirlwind sty...,sadness,joy,0.295394
4,i feel sympathetic towards her she was tired a...,love,fear,0.297408
5,i feel like such a noob when the customers mak...,sadness,anger,0.300051
6,i see momo feel shy momo hmmm gt me heyy momo,fear,sadness,0.304680
7,i feel afraid agn lol whats new,fear,sadness,0.307365
8,i feel like in the last year especially i ve g...,fear,joy,0.315156
9,i have to take jenny in to be spayed so of cou...,fear,sadness,0.321151


In [12]:
cm = confusion_matrix(test_metrics['gold'], test_metrics['preds'])
plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix on Test Split')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.xticks(range(len(label_names)), label_names, rotation=45, ha='right')
plt.yticks(range(len(label_names)), label_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', color=color)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()


/tmp/ipykernel_84246/3162531733.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
plt.figure(figsize=(7, 4))
plt.plot(history_df['epoch'], history_df['train_loss'], marker='o', label='train_loss')
plt.plot(history_df['epoch'], history_df['val_loss'], marker='o', label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Curves')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'training_curves.png', dpi=160, bbox_inches='tight')
plt.show()


/tmp/ipykernel_84246/839881396.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)
worst_pair = np.unravel_index(np.argmax(off_diag), off_diag.shape)
worst_true = label_names[worst_pair[0]]
worst_pred = label_names[worst_pair[1]]
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
best_class = label_names[int(np.argmax(per_class_accuracy))]

summary = {
    'seed': SEED,
    'device': str(DEVICE),
    'official_sizes': official_sizes,
    'experiment_sizes': EXPERIMENT_SIZES,
    'label_names': label_names,
    'finetune_model_name': FINETUNE_MODEL_NAME,
    'inference_model_name': INFERENCE_MODEL_NAME,
    'max_length': MAX_LENGTH,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'best_epoch': best_epoch,
    'test_loss': test_metrics['loss'],
    'test_accuracy': test_metrics['accuracy'],
    'test_f1_macro': test_metrics['f1_macro'],
    'best_class': best_class,
    'worst_true': worst_true,
    'worst_pred': worst_pred,
}
(ARTIFACTS_DIR / 'results_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'seed': 42,
 'device': 'cpu',
 'official_sizes': {'train': 16000, 'validation': 2000, 'test': 2000},
 'experiment_sizes': {'train': 4000, 'validation': 1000, 'test': 1000},
 'label_names': ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'],
 'finetune_model_name': 'google/bert_uncased_L-2_H-128_A-2',
 'inference_model_name': 'distilbert-base-uncased-finetuned-sst-2-english',
 'max_length': 128,
 'epochs': 5,
 'learning_rate': 5e-05,
 'best_epoch': 5,
 'test_loss': 0.7379539981484413,
 'test_accuracy': 0.786,
 'test_f1_macro': 0.6729319444952893,
 'best_class': 'joy',
 'worst_true': 'love',
 'worst_pred': 'joy'}